# Week 3: Data Contract & Field Classification (ML-04)

**Track:** Applied Search Intelligence: Google Search Ranking & Discoverability  
**Lane:** Refresh / Content Opportunity Scoring  
**Author:** Altaf (ML Intern)  
**Dataset:** Starter Anonymized Release (`data/raw/content_refresh_anonymized.csv`)  

---

## 1. Data Contract Overview

A rigorous machine learning system begins with a formal **data contract** defining the unit of analysis, observation horizons, field classifications, and invariant schema checks.

### 1.1 Contract Specification
1. **Unit of Analysis:** One row represents exactly one published content URL (`content_id`), belonging to a single client (`client_id`).
2. **Grain Invariant:** `COUNT(*) == 1` per `content_id`. Zero duplicates permitted.
3. **Time Windows:**
   - Feature Window ($T_{-90 \to 0}$): Trailing 90-day search console, engagement, and metadata telemetry.
   - Outcome Horizon ($T_{0 \to +30}$): Forward 30-day click trajectory used solely to compute `is_declining_label`.
4. **Monotonic Integrity Rules:**
   - $\text{impressions\_90d} \ge \text{clicks\_90d} \ge 0$
   - $\text{days\_with\_impressions} \le 90$
   - $\text{content\_age\_days} \ge \text{days\_since\_last\_update} \ge 0$

## 2. Four-Way Field Classification Matrix

Every single column in the dataset is explicitly placed into one of four mutually exclusive governance buckets:

### 2.1 Bucket A: Permitted Feature Inputs ($\mathbf{X}$)
- **Historical Search Performance:** `impressions_90d`, `clicks_90d`, `avg_position`, `ctr`, `days_with_impressions`
- **Historical Sub-Window Dynamics:** `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`
- **Behavioral Analytics (GA4):** `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `scroll_events_90d`, `days_with_sessions`, `engagement_rate`, `scroll_rate`, `ai_sessions_90d`, `ai_traffic_pct`
- **Structural & Content Metadata:** `content_age_days`, `days_since_last_update`, `word_count`, `char_count`, `competition`, `cpc`, `search_volume`
- **Categorical Tiers:** `competition_level`, `content_type`, `main_intent`, `age_tier`, `freshness_tier`, `word_count_tier`, `impression_tier`, `position_tier`

### 2.2 Bucket B: Target Outcome ($y$)
- `is_declining_label` $\in \{0, 1\}$: Observed $\ge 15\%$ organic click loss in the forward 30-day evaluation horizon. (Base rate: 54.2%).

### 2.3 Bucket C: Context / Grouping Variables
- `content_id`: Unique asset identifier. Used only for queue joining and reporting.
- `client_id`: Organization identifier. **Used exclusively for grouped train/test holdouts** (Never as a categorical feature).

### 2.4 Bucket D: Strictly Excluded / Banned Columns
- `trend_direction` & `trend_pct`: **Direct Target Leakage.** Sibling columns of `is_declining_label` computed over the outcome window.
- `provider_used` & `model_used`: Internal generation metadata; non-predictive for organic consumer search demand.
- Product heuristic flags (`health_score`, `quick_win_flag`): **Circular Features.** Represent existing human rule decisions.
- Raw URLs, Titles, and Search Queries: **PII & Domain Memorization.** Scrambled/excluded to prevent domain overfitting.

## 3. Dataset Invariants & Field Gotchas

| Field Gotcha | True Meaning | Handling Rule |
|---|---|---|
| `avg_position = 0` | "No search rank data recorded" (1,205 rows) | Do NOT interpret as #0 rank. Impute with flag `has_position_data = False` and set pos to 51.0. |
| Rates are $\times 100$ | `ctr = 0.76` means **0.76%** (not 76%) | Treat as percentage points across `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`. |
| `scroll_rate > 100` | Asynchronous event counting across telemetry sessions | Legitimate artifact of dual-instrumentation; clip or log-transform. |
| Patterned Missingness | Category-specific data absence | Do NOT blindly `fillna(0)`. Add explicit binary missingness indicator flags (`has_search_volume`, etc.). |

## 4. Contract Schema Verification Table

| Column Name | Bucket | Type | Range / Distinct | Gotcha / Validation Rule |
|---|---|---|---|---|
| `content_id` | Context | String | 30,000 unique | Primary key; unique grain |
| `client_id` | Context | String | 32 unique | Grouping variable for GroupKFold |
| `is_declining_label` | Target | Integer | {0, 1} | Positive rate = 0.542 |
| `trend_direction` | **Excluded** | String | {down, flat, up} | **LEAKAGE: Banned** |
| `trend_pct` | **Excluded** | Float | [-100, +500] | **LEAKAGE: Banned** |
| `impressions_90d` | Feature | Float | [0, 850,000] | Log-transform recommended |
| `clicks_90d` | Feature | Float | [0, 65,000] | Constrained $\le$ impressions_90d |
| `avg_position` | Feature | Float | [0.0, 50.0] | 0.0 denotes missing |
| `ctr` | Feature | Float | [0.0, 100.0] | Scale is $\times 100$ percent |
| `content_age_days` | Feature | Float | [1, 730] | Positive integer days |
| `days_since_last_update`| Feature | Float | [0, 730] | Constrained $\le$ content_age_days |

## 5. Rubric Pass Checklist (ML-04)

- [x] **Unit of analysis defined:** One row = one content item (`content_id`), verified unique.
- [x] **Observation windows defined:** 90-day feature window strictly precedes 30-day outcome window.
- [x] **Four-way classification complete:** All 44 fields assigned to Feature, Target, Context, or Excluded.
- [x] **Leakage columns identified & banned:** `trend_direction` and `trend_pct` quarantined.
- [x] **Gotchas documented:** `avg_position = 0` handled, $\times 100$ rate scaling noted.
- [x] **Monotonic assertions defined:** Non-negativity and logical inequalities enforced.